File search is a tool available in the Responses API. 
It enables models to retrieve information in a knowledge base of previously uploaded files through semantic and keyword search. 
This is a hosted tool managed by OpenAI
When the model decides to use it, it will automatically call the tool, retrieve information from your files, and return an output.
https://developers.openai.com/api/docs/guides/tools-file-search

Must run all these steps together

### Step 1
Upload a file to the File API (use example file)

In [ ]:
# Assuming OPENAI_API_KEY is set as an environment variable
import requests
from io import BytesIO
from openai import OpenAI

client = OpenAI()


def create_file(client, file_path):
    if file_path.startswith("http://") or file_path.startswith("https://"):
        # Download the file content from the URL
        response = requests.get(file_path)
        file_content = BytesIO(response.content)
        file_name = file_path.split("/")[-1]
        file_tuple = (file_name, file_content)
        result = client.files.create(file=file_tuple, purpose="assistants")
    else:
        # Handle local file path
        with open(file_path, "rb") as file_content:
            result = client.files.create(file=file_content, purpose="assistants")
    print(result.id)
    return result.id


# Replace with your own file path or URL
# file_id = create_file(client, "https://cdn.openai.com/API/docs/deep_research_blog.pdf")
file_id = create_file(
    client, "/home/lisa/LLM-integration-play/SureAdhere Product principles.pdf"
)

### Step 2
Create a vector store and upload the file to it

In [ ]:
vector_store = client.vector_stores.create(name="knowledge_base")
print(vector_store.id)

result = client.vector_stores.files.create(
    vector_store_id=vector_store.id, file_id=file_id
)
print(result)

### Step 3
Once your knowledge base is set up, you can include the file_search tool in the list of tools available to the model, 
along with the list of vector stores in which to search.

In [ ]:
my_vector_store_id = result.vector_store_id
response = client.responses.create(
    model="gpt-5.5",
    input="Summarize the SureAdhere Product principles.pdf document in 3 bullet points.",
    tools=[
        {
            "type": "file_search",
            "vector_store_ids": [my_vector_store_id],
            "max_num_results": 2,
        }
    ],
)
# print(response)
print(response.output_text)